In [ ]:
import os
import zipfile
import cv2
import shutil
import random
from tqdm import tqdm
from google.colab import drive

In [ ]:
# 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
# --- [GROUND TRUTH EXTRACTION: CHOLECTRACK20] ---
import os
import json

# Correct path
TARGET_JSON = "/content/drive/MyDrive/datacholec/Training/VID02/vid02.json"

print(f" TARGETING AUTHENTIC GROUND TRUTH: {TARGET_JSON}\n")

if os.path.exists(TARGET_JSON):
    with open(TARGET_JSON, 'r') as f:
        data = json.load(f)

    print("--- RAW BBOX EXTRACTION ---")

    # Extracting the first available frame's annotation
    first_frame = list(data['annotations'].keys())[0]
    annotation_list = data['annotations'][first_frame]

    if annotation_list:
        first_instance = annotation_list[0]
        instrument_id = first_instance.get('instrument')
        bbox = first_instance.get('tool_bbox')

        print(f"Frame ID: {first_frame}")
        print(f"Instrument Class ID: {instrument_id}")
        print(f"Raw 'tool_bbox' vector: {bbox}")

        # Checking if it matches YOLO specification
        if bbox and len(bbox) == 5:
            print("\n ANALYSIS: The data is ALREADY in a 5-parameter format.")
            print("Next step: We just need to parse these exact arrays directly to .txt files.")
    else:
        print("No annotations found in the first frame.")
else:
    print(f" File not found. Please verify if the Drive is mounted and the path exists.")

In [ ]:
# --- [UNIVERSAL PARSER: MODULE 1 - CHOLECTRACK20] ---
import os
import json
from tqdm import tqdm

def parse_cholec_ground_truth(json_path, output_dir):
    """
    Extracts native normalized bounding boxes from CholecTrack20 JSONs
    and formats them strictly to the YOLO standard to avoid coordinate drift.
    """
    os.makedirs(output_dir, exist_ok=True)

    with open(json_path, 'r') as f:
        data = json.load(f)

    annotations = data.get('annotations', {})
    parsed_frames = 0

    print(f" Processing CholecTrack20 annotations from {os.path.basename(json_path)}...")

    for frame_id, instances in annotations.items():
        yolo_lines = []

        for instance in instances:
            class_id = instance.get('instrument')
            bbox = instance.get('tool_bbox') # Format is naturally [x_center, y_center, width, height]

            # Validating the presence of a complete bounding box array
            if class_id is not None and bbox and len(bbox) == 4:
                # Construct the pure YOLO string directly from the source JSON
                yolo_str = f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}"
                yolo_lines.append(yolo_str)

        # Save the properly formatted ground truth file if instances exist
        if yolo_lines:
            txt_filename = f"cholec_{frame_id}.txt"
            txt_path = os.path.join(output_dir, txt_filename)

            with open(txt_path, 'w') as f_out:
                f_out.write("\n".join(yolo_lines))

            parsed_frames += 1

    print(f" Successfully extracted and formatted {parsed_frames} frames to YOLO standard.")

# Execute the parser on the discovered ground truth
TARGET_JSON = "/content/drive/MyDrive/datacholec/Training/VID02/vid02.json"
CLEAN_OUTPUT = "/content/dataset_a100/clean_labels/cholec"

parse_cholec_ground_truth(TARGET_JSON, CLEAN_OUTPUT)

In [ ]:
# --- [UNIVERSAL PARSER: MODULE 2 - ROBUST-MIPS] ---
import os
import json
import zipfile

def parse_robust_keypoints_test(zip_path, max_samples=1):
    """
    Extracts Pose Estimation keypoints from ROBUST-MIPS and calculates
    the Enclosure Bounding Box, formatting it strictly to the YOLO standard.
    """
    print(f" Processing ROBUST-MIPS from {os.path.basename(zip_path)}...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Find all toolposes.json files inside the ZIP
        pose_files = [f for f in z.namelist() if f.endswith('toolposes.json') and '__MACOSX' not in f]

        if not pose_files:
            print(" No toolposes.json found in the ZIP.")
            return

        print(f" Found {len(pose_files)} annotation files. Testing mathematical translation on the first one...")

        # Original ROBUST-MIPS resolution from Data Understanding
        img_w, img_h = 960, 540

        for i in range(min(max_samples, len(pose_files))):
            target_file = pose_files[i]

            with z.open(target_file) as f:
                data = json.loads(f.read().decode('utf-8'))

            yolo_lines = []
            for instance in data:
                nodes = instance.get('nodes', [])
                # Filter out 'null' values (occluded or missing joints)
                valid_nodes = [n for n in nodes if n is not None]

                if valid_nodes:
                    # Extract X and Y arrays
                    xs = [n[0] for n in valid_nodes]
                    ys = [n[1] for n in valid_nodes]

                    # Calculate Absolute Enclosure Box
                    x_min, x_max = min(xs), max(xs)
                    y_min, y_max = min(ys), max(ys)

                    # Translate to YOLO Normalized Format
                    x_center = ((x_min + x_max) / 2) / img_w
                    y_center = ((y_min + y_max) / 2) / img_h
                    box_w = (x_max - x_min) / img_w
                    box_h = (y_max - y_min) / img_h

                    # Placeholder class '0' for the proof of concept
                    class_id = 0
                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")

            print(f"\n Source File: {target_file}")
            print(f" Output YOLO String: {yolo_lines[0] if yolo_lines else 'No valid nodes'}")
            print("\nIf the values above are decimals between 0 and 1, the translation is perfect!")

# Execute the proof of concept directly from the Drive
ROBUST_ZIP = "/content/drive/MyDrive/ROB.zip"

if os.path.exists(ROBUST_ZIP):
    parse_robust_keypoints_test(ROBUST_ZIP, max_samples=1)
else:
    print(f" ROBUST-MIPS ZIP not found at: {ROBUST_ZIP}")

In [ ]:
# --- [UNIVERSAL PARSER: MODULE 3 - BADILLA-SOLÓRZANO] ---
import os
import zipfile
import numpy as np
from PIL import Image
import io

def parse_badilla_masks_test(zip_path, max_samples=1):
    """
    Extracts Semantic Segmentation Masks from Badilla-Solórzano, finds the pixel
    boundaries of each tool class, and calculates the normalized YOLO Bounding Box.
    """
    print(f" Processing Badilla-Solórzano from {os.path.basename(zip_path)}...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Find all mask PNG files inside the ZIP
        all_files = z.namelist()
        mask_files = [f for f in all_files if 'masks/' in f.lower() and f.endswith('.png') and "__MACOSX" not in f]

        if not mask_files:
            print(" No masks found in the ZIP.")
            return

        print(f" Found {len(mask_files)} mask files. Testing pixel-to-YOLO math on the first one...")

        # Original Badilla resolution
        img_w, img_h = 640, 480

        for i in range(min(max_samples, len(mask_files))):
            target_file = mask_files[i]

            with z.open(target_file) as f:
                # Read image directly from RAM without extracting to disk
                img_bytes = f.read()
                img = Image.open(io.BytesIO(img_bytes))
                mask_array = np.array(img)

            # Find all unique pixel values (classes) in this image
            unique_classes = np.unique(mask_array)
            yolo_lines = []

            for class_id in unique_classes:
                if class_id == 0: # 0 is always the background (black)
                    continue

                # Locate every single pixel belonging to this specific tool
                y_indices, x_indices = np.where(mask_array == class_id)

                if len(x_indices) > 0 and len(y_indices) > 0:
                    # Find the extreme boundaries of the painted pixels
                    x_min, x_max = np.min(x_indices), np.max(x_indices)
                    y_min, y_max = np.min(y_indices), np.max(y_indices)

                    # Translate Absolute Boundaries to Normalized YOLO Format
                    x_center = ((x_min + x_max) / 2) / img_w
                    y_center = ((y_min + y_max) / 2) / img_h
                    box_w = (x_max - x_min) / img_w
                    box_h = (y_max - y_min) / img_h

                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")

            print(f"\n Source Mask: {target_file}")
            print(f"Classes found in this mask (0 is background): {unique_classes.tolist()}")
            if not yolo_lines:
                print(" No tools found in this specific mask (empty frame).")
            for line in yolo_lines:
                print(f" Output YOLO String: {line}")

            print("\nIf the values above are decimals between 0 and 1, the Badilla translation is perfect!")

# Execute the proof of concept directly from the Drive
BADILLA_ZIP = "/content/drive/MyDrive/dataset.zip"

if os.path.exists(BADILLA_ZIP):
    parse_badilla_masks_test(BADILLA_ZIP, max_samples=1)
else:
    print(f" Badilla ZIP not found at: {BADILLA_ZIP}")

In [ ]:
# --- [VISUAL SANITY CHECK: BADILLA-SOLÓRZANO] ---
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

BADILLA_ZIP = "/content/drive/MyDrive/dataset.zip"

print(" EXECUTING VISUAL SANITY CHECK: PIXEL TO YOLO ALIGNMENT...\n")

with zipfile.ZipFile(BADILLA_ZIP, 'r') as z:
    all_files = z.namelist()

    # The exact mask file that was successfully parsed in the test
    target_mask = "Deep-learning-based-instrument-detection-for-intra-operative-robotic-assistance-main/Datasets/test/masks/im_1046.png"

    # Dynamically locating the corresponding real surgical image
    target_img = target_mask.replace('/masks/', '/images/')

    # Handling potential extension differences (png vs jpg)
    if target_img not in all_files:
        target_img = target_img.replace('.png', '.jpg')

    if target_mask in all_files and target_img in all_files:
        print(f" Found matching real image: {target_img.split('/')[-1]}")

        # 1. Read the Semantic Mask
        mask_bytes = z.read(target_mask)
        mask_array = np.array(Image.open(io.BytesIO(mask_bytes)))

        # 2. Read the Real Surgical Image
        img_bytes = z.read(target_img)
        img_array = np.array(Image.open(io.BytesIO(img_bytes)))

        # Ensure image is in RGB format for OpenCV drawing
        if len(img_array.shape) == 2:
            img_rgb = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        else:
            img_rgb = img_array.copy()

        unique_classes = np.unique(mask_array)

        # 3. Calculate and Draw Boundaries
        for class_id in unique_classes:
            if class_id == 0:
                continue # Ignore black background

            y_indices, x_indices = np.where(mask_array == class_id)
            if len(x_indices) > 0 and len(y_indices) > 0:
                # Absolute Boundaries
                x_min, x_max = np.min(x_indices), np.max(x_indices)
                y_min, y_max = np.min(y_indices), np.max(y_indices)

                # Drawing the Enclosure Box directly on the real image
                cv2.rectangle(img_rgb, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)

                # Adding the Class Label above the box
                label = f"Class {class_id}"
                cv2.putText(img_rgb, label, (x_min, max(20, y_min - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # 4. Render the Result
        plt.figure(figsize=(12, 8))
        plt.imshow(img_rgb)
        plt.title("Badilla Ground Truth Validation (Mask to YOLO BBox)")
        plt.axis('off')
        plt.show()

        print(" Visual plot generated successfully!")
    else:
        print(f" Error: Could not find the corresponding real image inside the ZIP.")
        print(f"Expected path: {target_img}")

In [ ]:
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

BADILLA_ZIP = "/content/drive/MyDrive/dataset.zip"

print(" DEBUGGING ZIP STRUCTURE...")

with zipfile.ZipFile(BADILLA_ZIP, 'r') as z:
    all_files = z.namelist()

    # 1. Let's find where the images actually are
    potential_images = [f for f in all_files if 'im_1046' in f and 'masks' not in f.lower() and '__MACOSX' not in f]
    potential_masks = [f for f in all_files if 'im_1046' in f and 'masks' in f.lower() and '__MACOSX' not in f]

    if not potential_images or not potential_masks:
        print(" Could not find the pair for im_1046. Listing some files to help:")
        print(all_files[:20]) # Show first 20 files to understand the path
    else:
        target_img_path = potential_images[0]
        target_mask_path = potential_masks[0]

        print(f" Found Image: {target_img_path}")
        print(f" Found Mask: {target_mask_path}")

        # --- START VISUAL CHECK ---
        mask_array = np.array(Image.open(io.BytesIO(z.read(target_mask_path))))
        img_array = np.array(Image.open(io.BytesIO(z.read(target_img_path))))

        # Convert to RGB for plotting
        if len(img_array.shape) == 2:
            img_display = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
        else:
            img_display = img_array.copy()

        # Original Badilla resolution (adjusting dynamically to the file found)
        img_h, img_w = mask_array.shape[:2]

        unique_classes = np.unique(mask_array)
        for class_id in unique_classes:
            if class_id == 0: continue

            y_indices, x_indices = np.where(mask_array == class_id)
            if len(x_indices) > 0:
                x_min, x_max = np.min(x_indices), np.max(x_indices)
                y_min, y_max = np.min(y_indices), np.max(y_indices)

                # Draw on image
                cv2.rectangle(img_display, (x_min, y_min), (x_max, y_max), (255, 0, 0), 3)
                cv2.putText(img_display, f"Class {class_id}", (x_min, y_min-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        plt.figure(figsize=(10, 6))
        plt.imshow(img_display)
        plt.title(f"Visual Validation: {os.path.basename(target_img_path)}")
        plt.axis('off')
        plt.show()

In [ ]:
# --- [UNIVERSAL PARSER: MODULE 3 - BADILLA-SOLÓRZANO VISUAL CHECK] ---
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

def validate_badilla_mapping(zip_path, target_filename='da_0_im_10.png'):
    """
    Locates a specific mask in the Badilla-Solórzano dataset, calculates
    YOLO-style bounding boxes from pixel clusters, and overlays them
    on the image for visual verification.
    """
    print(f" SEARCHING FOR TARGET: {target_filename}...")

    if not os.path.exists(zip_path):
        print(f" ERROR: Zip file not found at {zip_path}")
        return

    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()

        # Finding the full path of the target file within the nested ZIP structure
        match = [f for f in all_files if target_filename in f and '__MACOSX' not in f]

        if not match:
            print(f" ERROR: Could not find {target_filename} inside the ZIP.")
            # Fallback: list some files to help debugging
            print("Available files in baseline/ folder:")
            print([f for f in all_files if 'baseline/' in f][:5])
            return

        target_path = match[0]
        print(f" FILE LOCATED: {target_path}")

        # 1. Load the image/mask
        # Note: In the baseline folder of this dataset, masks are often stored as
        # grayscale images where pixel value = class ID.
        file_bytes = z.read(target_path)
        img = Image.open(io.BytesIO(file_bytes))
        mask_array = np.array(img)

        # 2. Prepare display image (Convert to RGB for colored boxes)
        if len(mask_array.shape) == 2:
            display_img = cv2.cvtColor(mask_array, cv2.COLOR_GRAY2RGB)
            # Enhance visibility if the image is too dark (optional)
            display_img = cv2.normalize(display_img, None, 0, 255, cv2.NORM_MINMAX)
        else:
            display_img = mask_array.copy()

        # 3. Extract Unique Classes (excluding background 0)
        unique_classes = np.unique(mask_array)
        print(f"Detected Class IDs in this frame: {unique_classes}")

        # 4. Calculate Bounding Boxes from Pixels
        found_objects = False
        for class_id in unique_classes:
            if class_id == 0 or class_id > 50: # Ignore background and noise
                continue

            # Find coordinates of all pixels belonging to this class
            y_indices, x_indices = np.where(mask_array == class_id)

            if len(x_indices) > 0:
                found_objects = True
                # Get the absolute min/max to define the box
                x_min, x_max = np.min(x_indices), np.max(x_indices)
                y_min, y_max = np.min(y_indices), np.max(y_indices)

                # Draw the rectangle on the display image
                # Color: Red (255, 0, 0), Thickness: 2
                cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

                # Label the box
                label = f"Class {class_id}"
                cv2.putText(display_img, label, (x_min, max(15, y_min - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        # 5. Render the result
        if found_objects:
            plt.figure(figsize=(10, 7))
            plt.imshow(display_img)
            plt.title(f"Visual Validation: {target_filename}")
            plt.axis('off')
            plt.show()
            print(" Visual Sanity Check complete. Verify if boxes tightly fit the tools.")
        else:
            print(" No tool classes found in this specific mask (it might be an empty frame).")

# EXECUTION
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
validate_badilla_mapping(ZIP_PATH)

In [ ]:
# --- [UNIVERSAL PARSER: MODULE 3 - BADILLA-SOLÓRZANO V2] ---
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

def validate_badilla_v2(zip_path, target_filename='da_0_im_10.png'):
    """
    Handles 3D arrays and filters pixel noise to extract clean YOLO boxes.
    """
    print(f" ANALYZING: {target_filename}...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()
        match = [f for f in all_files if target_filename in f and '__MACOSX' not in f]

        if not match:
            print(" File not found.")
            return

        # 1. Load and force conversion to Grayscale ('L' mode)
        file_bytes = z.read(match[0])
        img_raw = Image.open(io.BytesIO(file_bytes)).convert('L')
        mask_array = np.array(img_raw)

        # 2. Create RGB version for visualization
        display_img = cv2.cvtColor(mask_array, cv2.COLOR_GRAY2RGB)

        # 3. Identify actual tool clusters
        # Since the previous run showed 0-255, we only look for dominant pixel values
        unique_classes = np.unique(mask_array)

        found_objects = False
        for class_id in unique_classes:
            if class_id == 0 or class_id == 255: # Skip background and pure white noise
                continue

            # Now np.where will correctly unpack only 2 values (y, x)
            y_indices, x_indices = np.where(mask_array == class_id)

            # Only consider a cluster if it has enough pixels to be a tool
            if len(x_indices) > 500:
                found_objects = True
                x_min, x_max = np.min(x_indices), np.max(x_indices)
                y_min, y_max = np.min(y_indices), np.max(y_indices)

                cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)
                cv2.putText(display_img, f"Tool ID: {class_id}", (x_min, y_min-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        if found_objects:
            plt.figure(figsize=(10, 6))
            plt.imshow(display_img)
            plt.title(f"Refined Validation: {target_filename}")
            plt.axis('off')
            plt.show()
        else:
            print(" The file analyzed appears to be a raw image, not a segmentation mask.")
            print("Action: We need to locate the '/masks/' folder inside the ZIP specifically.")

# EXECUTION
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
validate_badilla_v2(ZIP_PATH)

In [ ]:
# --- [VISUAL VALIDATION: BADILLA-SOLÓRZANO MASKS] ---
import os
import zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

def visual_check_badilla(zip_path):
    print(" STEP 1: VALIDATING BADILLA-SOLÓRZANO...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()

        # Filtering specifically for masks
        mask_files = [f for f in all_files if 'masks/' in f.lower() and f.endswith('.png') and "__MACOSX" not in f]

        if not mask_files:
            print(" No files found in /masks/ folder. Checking structure...")
            print(all_files[:10])
            return

        # Let's take a sample mask
        sample_mask_path = mask_files[min(10, len(mask_files)-1)]
        print(f" Testing Mask: {sample_mask_path}")

        # Load mask
        mask_bytes = z.read(sample_mask_path)
        mask_img = Image.open(io.BytesIO(mask_bytes)).convert('L')
        mask_array = np.array(mask_img)

        # Find the corresponding original image (replacing 'masks' with 'images')
        img_path = sample_mask_path.replace('masks/', 'images/').replace('mask', 'im') # Adjusting common naming patterns

        # If the path logic fails, we use the mask itself to see the shapes
        display_img = cv2.cvtColor(mask_array, cv2.COLOR_GRAY2RGB)
        display_img = cv2.normalize(display_img, None, 0, 255, cv2.NORM_MINMAX)

        unique_ids = np.unique(mask_array)
        print(f"Identified Tool IDs: {unique_ids}")

        for tool_id in unique_ids:
            if tool_id == 0: continue # Skip background

            y_idx, x_idx = np.where(mask_array == tool_id)
            if len(x_idx) > 0:
                x_min, x_max = np.min(x_idx), np.max(x_idx)
                y_min, y_max = np.min(y_idx), np.max(y_idx)

                # Draw Box
                cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
                cv2.putText(display_img, f"ID {tool_id}", (x_min, y_min-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        plt.figure(figsize=(10, 6))
        plt.imshow(display_img)
        plt.title("Badilla-Solórzano: Tool Delimitation Check")
        plt.axis('off')
        plt.show()

# EXECUTION
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"
visual_check_badilla(ZIP_PATH)

In [ ]:
# --- [VISUAL VALIDATION: ROBUST-MIPS KEYPOINTS TO BBOX] ---
import os
import zipfile
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import io

def visual_check_robust(zip_path):
    print(" STEP 2: VALIDATING ROBUST-MIPS...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = z.namelist()

        # 1. Locate toolposes.json and its corresponding image
        # Using the path identified in the Data Understanding report
        pose_files = [f for f in all_files if 'toolposes.json' in f and '__MACOSX' not in f]

        if not pose_files:
            print(" No toolposes.json found.")
            return

        # Let's pick a sample from the testing set
        sample_json_path = pose_files[0]
        # In ROBUST, the image is usually in the same folder or parent folder
        # Logic: ROBUST-MIPS/.../123000/toolposes.json -> ROBUST-MIPS/.../123000/frame.png
        sample_img_path = sample_json_path.replace('toolposes.json', 'frame.png')

        if sample_img_path not in all_files:
            # Fallback: search for any png in the same directory
            dir_prefix = os.path.dirname(sample_json_path)
            potential_imgs = [f for f in all_files if f.startswith(dir_prefix) and f.endswith('.png')]
            if potential_imgs: sample_img_path = potential_imgs[0]

        print(f" Testing Pair:\n   JSON: {sample_json_path}\n   IMG:  {sample_img_path}")

        # 2. Load Data
        with z.open(sample_json_path) as f:
            poses = json.load(f)

        img_bytes = z.read(sample_img_path)
        img = np.array(Image.open(io.BytesIO(img_bytes)))

        # Ensure RGB for drawing
        if len(img.shape) == 2:
            display_img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            display_img = img.copy()

        # 3. Calculate Enclosure Box from Nodes
        for tool in poses:
            nodes = tool.get('nodes', [])
            valid_nodes = [n for n in nodes if n is not None]

            if valid_nodes:
                xs = [n[0] for n in valid_nodes]
                ys = [n[1] for n in valid_nodes]

                # Absolute pixel coordinates
                x_min, x_max = int(min(xs)), int(max(xs))
                y_min, y_max = int(min(ys)), int(max(ys))

                # Draw the box and the points (XAI: See if nodes are inside the tool)
                cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 3)
                for node in valid_nodes:
                    cv2.circle(display_img, (int(node[0]), int(node[1])), 5, (255, 0, 0), -1)

        plt.figure(figsize=(10, 6))
        plt.imshow(display_img)
        plt.title("ROBUST-MIPS: Node-to-BBox Validation")
        plt.axis('off')
        plt.show()
        print(" Blue dots = Original Nodes | Green Box = Calculated YOLO Target")

# EXECUTION
ROB_ZIP_PATH = "/content/drive/MyDrive/ROB.zip"
visual_check_robust(ROB_ZIP_PATH)

In [ ]:
# --- [VISUAL VALIDATION: CHOLECTRACK20 GROUND TRUTH] ---
import os
import json
import cv2
import matplotlib.pyplot as plt
from PIL import Image

def visual_check_cholec(json_path, img_folder):
    print(" STEP 3: VALIDATING CHOLECTRACK20...")

    with open(json_path, 'r') as f:
        data = json.load(f)

    # Pick the first frame with annotations
    target_frame = list(data['annotations'].keys())[0]
    annotations = data['annotations'][target_frame]

    # In Cholec, images are named as 006701.png (6 digits)
    img_filename = f"{target_frame.zfill(6)}.png"
    img_path = os.path.join(img_folder, img_filename)

    if not os.path.exists(img_path):
        print(f" Image not found at: {img_path}")
        return

    # Load image and get dimensions
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_orig, w_orig = img.shape[:2]

    print(f" Testing Frame: {img_filename} ({w_orig}x{h_orig})")

    for inst in annotations:
        c_id = inst.get('instrument')
        bbox = inst.get('tool_bbox') # [x_center, y_center, width, height]

        if bbox and len(bbox) == 4:
            # Reconstructing pixels from normalized YOLO format
            # Coordinates are: center_x, center_y, width, height
            x_c, y_c, w_b, h_b = bbox

            x1 = int((x_c - w_b/2) * w_orig)
            y1 = int((y_c - h_b/2) * h_orig)
            x2 = int((x_c + w_b/2) * w_orig)
            y2 = int((y_c + h_b/2) * h_orig)

            # Draw on image
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 255, 0), 3)
            cv2.putText(img_rgb, f"Tool {c_id}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(img_rgb)
    plt.title(f"CholecTrack20 Validation: Frame {target_frame}")
    plt.axis('off')
    plt.show()

# EXECUTION
CHOLEC_JSON = "/content/drive/MyDrive/datacholec/Training/VID02/vid02.json"
CHOLEC_FRAMES = "/content/drive/MyDrive/datacholec/Training/VID02/Frames"
visual_check_cholec(CHOLEC_JSON, CHOLEC_FRAMES)

In [ ]:
# --- [CHOLECTRACK20 MULTI-FORMAT DIAGNOSTIC] ---
import os
import json
import cv2
import matplotlib.pyplot as plt

def diagnostic_cholec(json_path, img_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Target frame from the previous fail: 6701
    target_id = "6701"
    annotations = data['annotations'].get(target_id, [])

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    print(f" Diagnosing Frame {target_id} | Resolution: {w}x{h}")

    for inst in annotations:
        bbox = inst.get('tool_bbox')
        if len(bbox) >= 4:
            # We will test 3 different interpretations:
            # 1. YOLO Standard (Center X, Center Y, W, H) - Current (Yellow)
            # 2. Top-Left Format (Xmin, Ymin, W, H) - (Cyan)
            # 3. Corner Format (Xmin, Ymin, Xmax, Ymax) - (Magenta)

            b = bbox[-4:] # Take last 4 values

            # YELLOW: [x_center, y_center, width, height]
            xc, yc, wb, hb = b
            yolo_x1, yolo_y1 = int((xc - wb/2)*w), int((yc - hb/2)*h)
            yolo_x2, yolo_y2 = int((xc + wb/2)*w), int((yc + hb/2)*h)

            # CYAN: [x_min, y_min, width, height]
            xmin, ymin, wb2, hb2 = b
            tl_x1, tl_y1 = int(xmin*w), int(ymin*h)
            tl_x2, tl_y2 = int((xmin + wb2)*w), int((ymin + hb2)*h)

            # MAGENTA: [x_min, y_min, x_max, y_max]
            xm1, ym1, xm2, ym2 = b
            c_x1, c_y1 = int(xm1*w), int(ym1*h)
            c_x2, c_y2 = int(xm2*w), int(ym2*h)

            cv2.rectangle(img_rgb, (yolo_x1, yolo_y1), (yolo_x2, yolo_y2), (255, 255, 0), 4) # Yellow
            cv2.rectangle(img_rgb, (tl_x1, tl_y1), (tl_x2, tl_y2), (0, 255, 255), 3) # Cyan
            cv2.rectangle(img_rgb, (c_x1, c_y1), (c_x2, c_y2), (255, 0, 255), 2) # Magenta

    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.title("Diagnostic: Yellow=YOLO | Cyan=TopLeft | Magenta=Corners")
    plt.axis('off')
    plt.show()

# EXECUTION
JSON = "/content/drive/MyDrive/datacholec/Training/VID02/vid02.json"
IMG = "/content/drive/MyDrive/datacholec/Training/VID02/Frames/006701.png"
diagnostic_cholec(JSON, IMG)

In [ ]:
# --- [CORRECTED MODULE 1: CHOLECTRACK20 WITH CLASS MAPPING] ---
import os
import json

def finalize_cholec_labels_v2(json_path, output_dir):
    """
    Converts Top-Left [xmin, ymin, w, h] to YOLO [x_center, y_center, w, h]
    and applies the Universal Taxonomy.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Universal Taxonomy Mapping
    # Mapping original Cholec IDs to our 7-class standard
    # Cholec: 0:Grasper, 1:Bipolar, 2:Hook, 3:Scissors, 4:Clipper, 5:Irrigator, 6:SpecimenBag
    UNIVERSAL_MAP = {0: 0, 1: 5, 2: 1, 3: 2, 4: 3, 5: 4, 6: 6}

    with open(json_path, 'r') as f:
        data = json.load(f)

    annotations = data.get('annotations', {})
    print(f" Processing {len(annotations)} frames for CholecTrack20...")

    for frame_id, instances in annotations.items():
        yolo_entries = []
        for inst in instances:
            original_id = inst.get('instrument')
            bbox = inst.get('tool_bbox') # [xmin, ymin, width, height]

            # Apply class mapping to align with Universal Taxonomy
            new_class_id = UNIVERSAL_MAP.get(original_id)

            if new_class_id is not None and bbox and len(bbox) == 4:
                xmin, ymin, w_box, h_box = bbox

                # MATHEMATICAL CONVERSION TO YOLO (X_CENTER, Y_CENTER)
                # Corrected variables to fix NameError
                x_center = xmin + (w_box / 2)
                y_center = ymin + (h_box / 2)

                yolo_str = f"{new_class_id} {x_center:.6f} {y_center:.6f} {w_box:.6f} {h_box:.6f}"
                yolo_entries.append(yolo_str)

        if yolo_entries:
            filename = f"{frame_id.zfill(6)}.txt"
            with open(os.path.join(output_dir, filename), 'w') as f_out:
                f_out.write("\n".join(yolo_entries))

    print(" CholecTrack20 labels are now corrected, mapped, and YOLO-compliant.")

# EXECUTION
RAW_JSON = "/content/drive/MyDrive/datacholec/Training/VID02/vid02.json"
OUTPUT = "/content/dataset_a100/labels/cholec"
finalize_cholec_labels_v2(RAW_JSON, OUTPUT)

In [ ]:
# --- [THE GRAND UNIFICATION: MULTI-DATASET BALANCER] ---
import os
import cv2

class SurgicalDataFactory:
    def __init__(self, output_base="/content/dataset_a100"):
        self.output_base = output_base
        self.img_size = 640 # Universal resolution for YOLOv8/v11

        # Universal Taxonomy Mapping
        self.maps = {
            'cholec': {0:0, 1:5, 2:1, 3:2, 4:3, 5:4, 6:6},
            'badilla': {1:0, 3:1, 4:2, 14:3, 15:4, 8:5, 21:6},
            'robust': {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6}
        }

        os.makedirs(f"{output_base}/images/train", exist_ok=True)
        os.makedirs(f"{output_base}/labels/train", exist_ok=True)

    def process_and_resize(self, image, bboxes, dataset_type, filename):
        """
        Resizes image to 640x640 and normalizes coordinates.
        This ensures cross-hospital consistency.
        """
        h_orig, w_orig = image.shape[:2]
        resized_img = cv2.resize(image, (self.img_size, self.img_size))

        yolo_labels = []
        for box in bboxes:
            # class_id, x, y, w, h (depending on dataset format)
            orig_id, x, y, bw, bh = box
            new_id = self.maps[dataset_type].get(orig_id)

            if new_id is not None:
                # Normalization logic goes here (Top-Left for Cholec, etc.)
                # For now, we use the validated Top-Left to Center conversion
                yolo_labels.append(f"{new_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}")

        # Saving to the A100 fast storage
        cv2.imwrite(f"{self.output_base}/images/train/{filename}.jpg", resized_img)
        with open(f"{self.output_base}/labels/train/{filename}.txt", 'w') as f:
            f.write("\n".join(yolo_labels))

print(" Unified Data Factory initialized. Ready to merge 8,000 surgical frames.")

In [ ]:
# --- [SURGICAL AI: SSD CLEANUP UTILITY] ---
import os
import shutil

def purge_unused_folders():
    # List of folders to be deleted
    trash_list = [
        'production_data', 'production_dataset', 'raw_badilla',
        'raw_data', 'raw_robust', 'temp_labels', 'temp_labels_raw',
        'sample_data'
    ]

    print(" Starting Surgical AI Environment Cleanup...")

    for folder in trash_list:
        folder_path = os.path.join('/content', folder)
        if os.path.exists(folder_path):
            try:
                shutil.rmtree(folder_path)
                print(f" Deleted: {folder}")
            except Exception as e:
                print(f" Error deleting {folder}: {e}")
        else:
            print(f"ℹ Folder already gone: {folder}")

    # Ensure our target directory is clean and ready
    target_dir = '/content/dataset_a100'
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    os.makedirs(target_dir, exist_ok=True)
    print(f"\n ENVIRONMENT READY! Target: {target_dir}")

# EXECUTION
purge_unused_folders()

In [ ]:
# --- [SURGICAL AI: COMPREHENSIVE SSD CLEANUP] ---
import os
import shutil

def purge_all_legacy_data():
    """
    Removes all legacy and temporary folders to prevent data corruption
    during the final unification process on the A100 SSD.
    """
    # Adding 'final_training_data' and other suspected legacy folders
    trash_list = [
        'production_data', 'production_dataset', 'raw_badilla',
        'raw_data', 'raw_robust', 'temp_labels', 'temp_labels_raw',
        'sample_data', 'final_training_data'
    ]

    print(" Starting Deep Cleanup of Colab Environment...")

    for folder in trash_list:
        folder_path = os.path.join('/content', folder)
        if os.path.exists(folder_path):
            try:
                shutil.rmtree(folder_path)
                print(f" Removed Legacy Folder: {folder}")
            except Exception as e:
                print(f" Warning: Could not delete {folder} -> {e}")
        else:
            print(f"ℹ {folder} is already clean.")

    # Resetting the primary target directory
    target_dir = '/content/dataset_a100'
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    os.makedirs(target_dir, exist_ok=True)
    print(f"\n DISK PURIFIED! Target directory ready at: {target_dir}")

# EXECUTE BEFORE THE UNIFICATION SCRIPT
purge_all_legacy_data()

In [ ]:
# --- [SURGICAL DATA FACTORY: THE UNIFICATION SCRIPT] ---
import os
import cv2
import json
import zipfile
import numpy as np
from PIL import Image
import io
from tqdm import tqdm

class SurgicalDataFactory:
    def __init__(self, output_base="/content/dataset_a100"):
        self.output_base = output_base
        self.img_size = 640

        # Universal Taxonomy Mapping
        self.maps = {
            'cholec': {0:0, 1:5, 2:1, 3:2, 4:3, 5:4, 6:6},
            'badilla': {1:0, 3:1, 4:2, 14:3, 15:4, 8:5, 21:6},
            'robust': {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6}
        }

        # Create directory structure for YOLO
        for folder in ['images/train', 'labels/train']:
            os.makedirs(os.path.join(output_base, folder), exist_ok=True)

    def save_entry(self, img, yolo_labels, prefix, frame_id):
        """Standardizes saving format for YOLOv11."""
        if not yolo_labels: return

        filename = f"{prefix}_{frame_id}"
        img_path = os.path.join(self.output_base, f"images/train/{filename}.jpg")
        lbl_path = os.path.join(self.output_base, f"labels/train/{filename}.txt")

        cv2.imwrite(img_path, cv2.resize(img, (self.img_size, self.img_size)))
        with open(lbl_path, 'w') as f:
            f.write("\n".join(yolo_labels))

    # --- MODULE 1: CHOLECTRACK20 (TOP-LEFT FORMAT) ---
    def process_cholec(self, json_path, img_dir, quota=4000):
        print(f" Extracting {quota} frames from CholecTrack20...")
        with open(json_path, 'r') as f:
            data = json.load(f)

        frames = list(data['annotations'].keys())[:quota]
        for f_id in tqdm(frames):
            img_file = os.path.join(img_dir, f"{f_id.zfill(6)}.png")
            if not os.path.exists(img_file): continue

            img = cv2.imread(img_file)
            h, w = img.shape[:2]
            labels = []
            for inst in data['annotations'][f_id]:
                new_id = self.maps['cholec'].get(inst['instrument'])
                bbox = inst['tool_bbox'] # [xmin, ymin, width, height]
                if new_id is not None and len(bbox) == 4:
                    # Convert to YOLO center-based format
                    xc = (bbox[0] + bbox[2]/2)
                    yc = (bbox[1] + bbox[3]/2)
                    labels.append(f"{new_id} {xc:.6f} {yc:.6f} {bbox[2]:.6f} {bbox[3]:.6f}")
            self.save_entry(img, labels, "cholec", f_id)

    # --- MODULE 2: ROBUST-MIPS (KEYPOINTS/NODES FORMAT) ---
    def process_robust(self, zip_path, quota=2500):
        print(f" Extracting {quota} frames from ROBUST-MIPS...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            pose_files = [f for f in z.namelist() if 'toolposes.json' in f][:quota]
            for p_file in tqdm(pose_files):
                img_file = p_file.replace('toolposes.json', 'frame.png')
                if img_file not in z.namelist(): continue

                poses = json.loads(z.read(p_file))
                img = np.array(Image.open(io.BytesIO(z.read(img_file))))
                h, w = img.shape[:2]
                labels = []
                for tool in poses:
                    nodes = [n for n in tool.get('nodes', []) if n is not None]
                    if nodes:
                        xs, ys = [n[0] for n in nodes], [n[1] for n in nodes]
                        # Calculate Enclosure Box
                        xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
                        bw, bh = (xmax - xmin), (ymax - ymin)
                        xc, yc = (xmin + bw/2), (ymin + bh/2)
                        new_id = self.maps['robust'].get(tool.get('class_id', 1)) # Default to 1
                        labels.append(f"{new_id} {xc/w:.6f} {yc/h:.6f} {bw/w:.6f} {bh/h:.6f}")
                self.save_entry(img, labels, "robust", p_file.split('/')[-2])

    # --- MODULE 3: BADILLA-SOLÓRZANO (MASK FORMAT) ---
    def process_badilla(self, zip_path, quota=1500):
        print(f" Extracting {quota} frames from Badilla-Solórzano...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            mask_files = [f for f in z.namelist() if 'masks/' in f and f.endswith('.png')][:quota]
            for m_file in tqdm(mask_files):
                img_file = m_file.replace('masks/', 'images/').replace('.png', '.jpg')
                if img_file not in z.namelist(): img_file = m_file.replace('masks/', 'images/')
                if img_file not in z.namelist(): continue

                mask = np.array(Image.open(io.BytesIO(z.read(m_file))).convert('L'))
                img = np.array(Image.open(io.BytesIO(z.read(img_file))))
                h, w = mask.shape[:2]
                labels = []
                for class_id in np.unique(mask):
                    if class_id == 0: continue
                    y_idx, x_idx = np.where(mask == class_id)
                    if len(x_idx) > 100: # Filter noise
                        xmin, xmax, ymin, ymax = np.min(x_idx), np.max(x_idx), np.min(y_idx), np.max(y_idx)
                        bw, bh = (xmax - xmin), (ymax - ymin)
                        xc, yc = (xmin + bw/2), (ymin + bh/2)
                        new_id = self.maps['badilla'].get(class_id)
                        if new_id is not None:
                            labels.append(f"{new_id} {xc/w:.6f} {yc/h:.6f} {bw/w:.6f} {bh/h:.6f}")
                self.save_entry(img, labels, "badilla", os.path.basename(m_file).split('.')[0])

# --- EXECUTION ENGINE ---
factory = SurgicalDataFactory()

# Paths (Using verified paths)
factory.process_cholec("/content/drive/MyDrive/datacholec/Training/VID02/vid02.json", "/content/drive/MyDrive/datacholec/Training/VID02/Frames", quota=4000)
factory.process_robust("/content/drive/MyDrive/ROB.zip", quota=2500)
factory.process_badilla("/content/drive/MyDrive/dataset.zip", quota=1500)

print("\n SUCCESS! Balanceated frames are ready in /content/dataset_a100")

In [ ]:
# --- [FINAL DATASET AUDIT: STATISTICS & VISUAL VERIFICATION] ---
import os
import cv2
import random
import matplotlib.pyplot as plt
from collections import Counter

def audit_dataset(base_path="/content/dataset_a100", num_samples=3):
    """
    Analyzes class distribution and performs visual sanity checks on the
    unified surgical dataset.
    """
    images_dir = os.path.join(base_path, "images/train")
    labels_dir = os.path.join(base_path, "labels/train")

    # Universal Taxonomy Labels
    class_names = [
        "Grasper", "Hook", "Scissors", "Clipper",
        "Irrigator", "Bipolar", "Specimen Bag"
    ]

    # 1. Statistical Analysis
    print(" ANALYZING CLASS DISTRIBUTION...")
    label_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
    all_instances = []

    for lbl in label_files:
        with open(os.path.join(labels_dir, lbl), 'r') as f:
            for line in f:
                class_id = int(line.split()[0])
                all_instances.append(class_id)

    stats = Counter(all_instances)
    print(f"\nTotal Frames: {len(label_files)}")
    print(f"Total Tool Instances: {len(all_instances)}")
    print("-" * 30)
    for i, name in enumerate(class_names):
        count = stats.get(i, 0)
        percentage = (count / len(all_instances) * 100) if all_instances else 0
        print(f"[{i}] {name.ljust(15)}: {count} ({percentage:.2f}%)")
    print("-" * 30)

    # 2. Visual Verification
    print(f"\n GENERATING {num_samples} RANDOM VISUAL CHECKS...")
    sample_labels = random.sample(label_files, min(num_samples, len(label_files)))

    fig, axes = plt.subplots(1, num_samples, figsize=(18, 6))
    if num_samples == 1: axes = [axes]

    for i, lbl_name in enumerate(sample_labels):
        # Match label to image
        img_name = lbl_name.replace('.txt', '.jpg')
        img_path = os.path.join(images_dir, img_name)
        lbl_path = os.path.join(labels_dir, lbl_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        with open(lbl_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.split()))
                cid, xc, yc, wb, hb = parts

                # Rescale YOLO normalized coordinates to pixel values
                x1 = int((xc - wb/2) * w)
                y1 = int((yc - hb/2) * h)
                x2 = int((xc + wb/2) * w)
                y2 = int((yc + hb/2) * h)

                # Draw Box and Label
                color = (255, 0, 0) # Red
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label_text = class_names[int(cid)]
                cv2.putText(img, label_text, (x1, max(20, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        axes[i].imshow(img)
        axes[i].set_title(f"Sample: {img_name}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# EXECUTION
audit_dataset(num_samples=4)

In [ ]:
# --- [SURGICAL AI: BULLETPROOF DATA UNIFICATION] ---
import os
import cv2
import json
import zipfile
import numpy as np
from PIL import Image
import io
from tqdm import tqdm

class FinalUnifiedFactory:
    def __init__(self, output_base="/content/dataset_a100"):
        self.output_base = output_base
        self.img_size = 640
        # Universal Taxonomy: 0:Grasper, 1:Hook, 2:Scissors, 3:Clipper, 4:Irrigator, 5:Bipolar, 6:SpecBag
        self.maps = {
            'cholec': {0:0, 1:5, 2:1, 3:2, 4:3, 5:4, 6:6},
            'badilla': {1:0, 3:1, 4:2, 14:3, 15:4, 8:5, 21:6},
            'robust': {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6}
        }
        for f in ['images/train', 'labels/train']: os.makedirs(os.path.join(output_base, f), exist_ok=True)

    def save(self, img, labels, prefix, fid):
        if not labels: return
        fname = f"{prefix}_{fid}"
        cv2.imwrite(os.path.join(self.output_base, f"images/train/{fname}.jpg"), cv2.resize(img, (640,640)))
        with open(os.path.join(self.output_base, f"labels/train/{fname}.txt"), 'w') as f: f.write("\n".join(labels))

    def process_cholec(self, json_p, img_d, q=4000):
        with open(json_p, 'r') as f: data = json.load(f)
        for fid in tqdm(list(data['annotations'].keys())[:q], desc="Cholec"):
            path = os.path.join(img_d, f"{fid.zfill(6)}.png")
            if os.path.exists(path):
                img = cv2.imread(path)
                lbls = [f"{self.maps['cholec'][i['instrument']]} {i['tool_bbox'][0]+i['tool_bbox'][2]/2:.6f} {i['tool_bbox'][1]+i['tool_bbox'][3]/2:.6f} {i['tool_bbox'][2]:.6f} {i['tool_bbox'][3]:.6f}" for i in data['annotations'][fid] if i['instrument'] in self.maps['cholec']]
                self.save(img, lbls, "cholec", fid)

    def process_robust(self, zip_p, q=2500):
        with zipfile.ZipFile(zip_p, 'r') as z:
            jsons = [f for f in z.namelist() if 'toolposes.json' in f and '__MACOSX' not in f][:q]
            for p in tqdm(jsons, desc="Robust"):
                # Search for any PNG in the same folder
                folder = os.path.dirname(p)
                imgs = [f for f in z.namelist() if f.startswith(folder) and f.endswith('.png')]
                if not imgs: continue
                img = np.array(Image.open(io.BytesIO(z.read(imgs[0]))))
                h, w = img.shape[:2]
                lbls = []
                for t in json.loads(z.read(p)):
                    nodes = [n for n in t.get('nodes', []) if n is not None]
                    if nodes:
                        xs, ys = [n[0] for n in nodes], [n[1] for n in nodes]
                        bw, bh = max(xs)-min(xs), max(ys)-min(ys)
                        lbls.append(f"0 {(min(xs)+bw/2)/w:.6f} {(min(ys)+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}") # Defaulting to Grasper for Robust Proof
                self.save(img, lbls, "robust", p.split('/')[-2])

    def process_badilla(self, zip_p, q=1500):
        with zipfile.ZipFile(zip_p, 'r') as z:
            masks = [f for f in z.namelist() if 'masks/' in f.lower() and f.endswith('.png')][:q]
            for m in tqdm(masks, desc="Badilla"):
                # Search for image in 'images' or 'baseline' sibling folders
                img_f = m.replace('masks/', 'images/').replace('.png', '.jpg')
                if img_f not in z.namelist(): img_f = m.replace('masks/', 'baseline/')
                if img_f not in z.namelist(): continue
                mask = np.array(Image.open(io.BytesIO(z.read(m))).convert('L'))
                img = np.array(Image.open(io.BytesIO(z.read(img_f))))
                h, w = mask.shape[:2]
                lbls = []
                for cid in np.unique(mask):
                    if cid == 0 or cid not in self.maps['badilla']: continue
                    y, x = np.where(mask == cid)
                    if len(x) > 100:
                        bw, bh = np.max(x)-np.min(x), np.max(y)-np.min(y)
                        lbls.append(f"{self.maps['badilla'][cid]} {(np.min(x)+bw/2)/w:.6f} {(np.min(y)+bh/2)/h:.6f} {bw/w:.6f} {bh/h:.6f}")
                self.save(img, lbls, "badilla", os.path.basename(m).split('.')[0])

# EXECUTION
factory = FinalUnifiedFactory()
factory.process_cholec("/content/drive/MyDrive/datacholec/Training/VID02/vid02.json", "/content/drive/MyDrive/datacholec/Training/VID02/Frames")
factory.process_robust("/content/drive/MyDrive/ROB.zip")
factory.process_badilla("/content/drive/MyDrive/dataset.zip")

In [ ]:
# --- [TEST 1: CHOLEC MATHEMATICS FIX] ---
import os, json, cv2, matplotlib.pyplot as plt
def quick_test_cholec(json_p, img_d):
    with open(json_p, 'r') as f: data = json.load(f)
    fid = "006701" # Example frame
    img = cv2.imread(os.path.join(img_d, f"{fid}.png"))
    h, w = img.shape[:2]
    # Universal Map: ID 3 (Cholec) -> ID 2 (Scissors)
    for inst in data['annotations'][fid.lstrip('0')]:
        b = inst['tool_bbox']
        # CORRECTION: b[0] is already normalized! We only multiplied by W for illustration purposes.
        x1, y1 = int(b[0]*w), int(b[1]*h)
        x2, y2 = int((b[0]+b[2])*w), int((b[1]+b[3])*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 0), 3)
        cv2.putText(img, f"Scissors (Mapped)", (x1, y1-10), 0, 0.7, (255,255,0), 2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Cholec Fix"); plt.show()

quick_test_cholec("/content/drive/MyDrive/datacholec/Training/VID02/vid02.json", "/content/drive/MyDrive/datacholec/Training/VID02/Frames")

In [ ]:
# --- [SURGICAL AI: ZIP STRUCTURE DIAGNOSTIC] ---
import zipfile

def check_my_zips(zip_path, name):
    print(f"\n---  ANALYZING: {name} ---")
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            # We took the first 20 files to understand the structure
            files = z.namelist()
            print(f"Total files found: {len(files)}")
            print("First 15 file paths found inside:")
            for f in files[:15]:
                print(f"  -> {f}")
    except Exception as e:
        print(f" Error opening {name}: {e}")

# SPIN FOR BOTH
check_my_zips("/content/drive/MyDrive/ROB.zip", "ROBUST-MIPS")
check_my_zips("/content/drive/MyDrive/dataset.zip", "BADILLA-SOLORZANO")

In [ ]:
# --- [SURGICAL AI: THE DEFINITIVE V6 UNIFIER] ---
import os, cv2, json, zipfile, shutil, numpy as np
from PIL import Image
import io
from tqdm import tqdm
import matplotlib.pyplot as plt

# 1. CLEANUP
target_path = '/content/dataset_a100'
if os.path.exists(target_path): shutil.rmtree(target_path)
for d in ['images/train', 'labels/train']: os.makedirs(os.path.join(target_path, d), exist_ok=True)

class FinalV6Factory:
    def __init__(self):
        # Universal Taxonomy: 0:Grasper, 1:Hook, 2:Scissors, 3:Clipper, 4:Irrigator, 5:Bipolar, 6:SpecBag
        self.maps = {
            'cholec': {0:0, 1:5, 2:1, 3:2, 4:3, 5:4, 6:6},
            'badilla': {1:0, 3:1, 4:2, 14:3, 15:4, 8:5, 21:6},
            'robust': {1:0, 2:1, 3:2, 4:3, 5:4, 6:5, 7:6}
        }

    def save(self, img, labels, prefix, fid):
        if not labels: return
        fname = f"{prefix}_{fid}"
        cv2.imwrite(f"{target_path}/images/train/{fname}.jpg", cv2.resize(img, (640,640)))
        with open(f"{target_path}/labels/train/{fname}.txt", 'w') as f: f.write("\n".join(labels))

    def process_cholec(self, json_p, img_d):
        with open(json_p, 'r') as f: data = json.load(f)
        for fid in tqdm(list(data['annotations'].keys()), desc="Cholec"):
            path = os.path.join(img_d, f"{fid.zfill(6)}.png")
            if not os.path.exists(path): continue
            lbls = [f"{self.maps['cholec'].get(i['instrument'])} {i['tool_bbox'][0]+i['tool_bbox'][2]/2:.6f} {i['tool_bbox'][1]+i['tool_bbox'][3]/2:.6f} {i['tool_bbox'][2]:.6f} {i['tool_bbox'][3]:.6f}"
                    for i in data['annotations'][fid] if i['instrument'] in self.maps['cholec']]
            self.save(cv2.imread(path), lbls, "cholec", fid)

    def process_robust(self, zip_p, q=2500):
        with zipfile.ZipFile(zip_p, 'r') as z:
            jsons = [f for f in z.namelist() if 'toolposes.json' in f and 'MACOSX' not in f][:q]
            for p in tqdm(jsons, desc="Robust"):
                img_f = p.replace('toolposes.json', 'raw.png') # FIXED NAME
                if img_f not in z.namelist(): continue
                img = np.array(Image.open(io.BytesIO(z.read(img_f))))
                h, w = img.shape[:2]
                lbls = []
                for t in json.loads(z.read(p)):
                    n = [node for node in t.get('nodes', []) if node is not None]
                    if n:
                        new_id = self.maps['robust'].get(t.get('class_id', 1), 0)
                        xs, ys = [pt[0] for pt in n], [pt[1] for pt in n]
                        bw, bh = (max(xs)-min(xs))/w, (max(ys)-min(ys))/h
                        lbls.append(f"{new_id} {(min(xs)/w + bw/2):.6f} {(min(ys)/h + bh/2):.6f} {bw:.6f} {bh:.6f}")
                self.save(img, lbls, "robust", p.split('/')[-2])

    def process_badilla(self, zip_p):
        with zipfile.ZipFile(zip_p, 'r') as z:
            # Finding masks using the real long path prefix
            prefix = "Deep-learning-based-instrument-detection-for-intra-operative-robotic-assistance-main/Datasets/"
            masks = [f for f in z.namelist() if 'masks/' in f and f.endswith('.png') and 'MACOSX' not in f]
            for m in tqdm(masks, desc="Badilla"):
                img_f = m.replace('masks/', 'baseline/') # FIXED FOLDER
                if img_f not in z.namelist(): continue
                mask = np.array(Image.open(io.BytesIO(z.read(m))).convert('L'))
                img = np.array(Image.open(io.BytesIO(z.read(img_f))))
                h, w = mask.shape[:2]
                lbls = []
                for cid in np.unique(mask):
                    if cid in self.maps['badilla']:
                        y, x = np.where(mask == cid)
                        bw, bh = (np.max(x)-np.min(x))/w, (np.max(y)-np.min(y))/h
                        lbls.append(f"{self.maps['badilla'][cid]} {(np.min(x)/w + bw/2):.6f} {(np.min(y)/h + bh/2):.6f} {bw:.6f} {bh:.6f}")
                self.save(img, lbls, "badilla", os.path.basename(m).split('.')[0])

# EXECUTION
v6 = FinalV6Factory()
v6.process_cholec("/content/drive/MyDrive/datacholec/Training/VID02/vid02.json", "/content/drive/MyDrive/datacholec/Training/VID02/Frames")
v6.process_robust("/content/drive/MyDrive/ROB.zip")
v6.process_badilla("/content/drive/MyDrive/dataset.zip")

In [ ]:
def verify_v6_matrix():
    classes = ["Grasper", "Hook", "Scissors", "Clipper", "Irrigator", "Bipolar", "Specimen Bag"]
    sources = ["cholec", "robust", "badilla"]
    matrix = {i: {src: None for src in sources} for i in range(7)}
    lbl_dir = "/content/dataset_a100/labels/train"
    for lbl in os.listdir(lbl_dir):
        src = next((s for s in sources if lbl.startswith(s)), None)
        if src:
            with open(os.path.join(lbl_dir, lbl), 'r') as f:
                cid = int(f.readline().split()[0])
                if cid < 7 and matrix[cid][src] is None: matrix[cid][src] = lbl

    fig, axes = plt.subplots(7, 3, figsize=(15, 25))
    for cid in range(7):
        for s_idx, src in enumerate(sources):
            ax = axes[cid, s_idx]; file = matrix[cid][src]
            if file:
                img = cv2.cvtColor(cv2.imread(f"/content/dataset_a100/images/train/{file.replace('.txt', '.jpg')}"), cv2.COLOR_BGR2RGB)
                with open(os.path.join(lbl_dir, file), 'r') as f:
                    for line in f:
                        p = list(map(float, line.split()))
                        if int(p[0]) == cid:
                            x1, y1 = int((p[1]-p[3]/2)*640), int((p[2]-p[4]/2)*640)
                            x2, y2 = int((p[1]+p[3]/2)*640), int((p[2]+p[4]/2)*640)
                            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 0), 4)
                ax.imshow(img); ax.set_title(f"{src.upper()} | {classes[cid]}")
            else: ax.text(0.5, 0.5, "NOT FOUND", ha='center')
            ax.axis('off')
    plt.tight_layout(); plt.show()

verify_v6_matrix()

In [ ]:
# --- [SURGICAL AI: BADILLA RESCUE & FINAL MATRIX] ---
import os, cv2, zipfile, numpy as np
from PIL import Image
import io
import matplotlib.pyplot as plt
from tqdm import tqdm

def rescue_badilla(zip_path="/content/drive/MyDrive/dataset.zip"):
    print(" Starting Badilla Rescue (Smart Name Search)...")
    maps_badilla = {1:0, 3:1, 4:2, 14:3, 15:4, 8:5, 21:6}
    target_path = '/content/dataset_a100'

    with zipfile.ZipFile(zip_path, 'r') as z:
        all_files = [f for f in z.namelist() if 'MACOSX' not in f]
        masks = [f for f in all_files if 'masks/' in f.lower() and f.endswith('.png')]

        # Create a dictionary with ALL images in the ZIP, ignoring folder structures
        img_files = [f for f in all_files if 'masks/' not in f.lower() and f.endswith(('.png', '.jpg'))]
        img_dict = {os.path.splitext(os.path.basename(f))[0]: f for f in img_files}

        count = 0
        for m in tqdm(masks, desc="Processing Badilla"):
            base_name = os.path.splitext(os.path.basename(m))[0] # e.g., 'im_10'

            # Smart match: search for the base name anywhere
            img_f = img_dict.get(base_name)
            if not img_f:
                # Fuzzy match in case it has a 'da_0_' prefix
                matched = [v for k, v in img_dict.items() if base_name in k]
                if matched: img_f = matched[0]
                else: continue

            mask = np.array(Image.open(io.BytesIO(z.read(m))).convert('L'))
            img = np.array(Image.open(io.BytesIO(z.read(img_f))))
            h, w = mask.shape[:2]
            lbls = []

            for cid in np.unique(mask):
                if cid in maps_badilla:
                    y, x = np.where(mask == cid)
                    if len(x) > 50:
                        bw, bh = (np.max(x)-np.min(x))/w, (np.max(y)-np.min(y))/h
                        xc, yc = (np.min(x)/w + bw/2), (np.min(y)/h + bh/2)
                        lbls.append(f"{maps_badilla[cid]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

            if lbls:
                fname = f"badilla_{base_name}"
                cv2.imwrite(f"{target_path}/images/train/{fname}.jpg", cv2.resize(img, (640,640)))
                with open(f"{target_path}/labels/train/{fname}.txt", 'w') as f:
                    f.write("\n".join(lbls))
                count += 1

    print(f" Badilla successfully rescued! {count} frames added.")

# 1. RUN RESCUE
rescue_badilla()

# 2. GENERATE FINAL MATRIX (ALL 3 DATASETS)
def show_final_matrix():
    classes = ["Grasper", "Hook", "Scissors", "Clipper", "Irrigator", "Bipolar", "Specimen Bag"]
    sources = ["cholec", "robust", "badilla"]
    matrix = {i: {src: None for src in sources} for i in range(7)}
    lbl_dir = "/content/dataset_a100/labels/train"

    for lbl in os.listdir(lbl_dir):
        src = next((s for s in sources if lbl.startswith(s)), None)
        if src:
            with open(os.path.join(lbl_dir, lbl), 'r') as f:
                cid = int(f.readline().split()[0])
                if cid < 7 and matrix[cid][src] is None: matrix[cid][src] = lbl

    fig, axes = plt.subplots(7, 3, figsize=(15, 25))
    for cid in range(7):
        for s_idx, src in enumerate(sources):
            ax = axes[cid, s_idx]; file = matrix[cid][src]
            if file:
                img_path = f"/content/dataset_a100/images/train/{file.replace('.txt', '.jpg')}"
                img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
                with open(os.path.join(lbl_dir, file), 'r') as f:
                    for line in f:
                        p = list(map(float, line.split()))
                        if int(p[0]) == cid:
                            x1, y1 = int((p[1]-p[3]/2)*640), int((p[2]-p[4]/2)*640)
                            x2, y2 = int((p[1]+p[3]/2)*640), int((p[2]+p[4]/2)*640)
                            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 0), 4)
                ax.imshow(img); ax.set_title(f"{src.upper()} | {classes[cid]}")
            else: ax.text(0.5, 0.5, "NOT FOUND", ha='center')
            ax.axis('off')
    plt.tight_layout(); plt.show()

show_final_matrix()

In [ ]:
# --- [BACKUP: SAVE TO GOOGLE DRIVE] ---
!cp -r /content/dataset_a100 /content/drive/MyDrive/DATASET_ICCSA_FINAL

In [ ]:
import os

# Point to the saved folder
lbl_dir = "/content/drive/MyDrive/DATASET_ICCSA_FINAL/labels/train"
allfiles = os.listdir(lbl_dir)

cholec_count = sum(1 for f in allfiles if f.startswith('cholec'))
robust_count = sum(1 for f in allfiles if f.startswith('robust'))
badilla_count = sum(1 for f in allfiles if f.startswith('badilla'))

print(" FINAL DATASET COUNT:")
print(f"Cholec: {cholec_count} frames")
print(f"Robust: {robust_count} frames")
print(f"Badilla: {badilla_count} frames")
print("-" * 30)
print(f"TOTAL: {len(allfiles)} unified frames!")